Dataset personalizado de PyTorch.

Carga de secuencias y etiquetas.

Tokenización de transcripciones.

Padding de secuencias por batch.

DataLoader y collate_fn.

Validación de longitudes para CTC.

In [8]:
import json
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

In [9]:
# Rutas
CLIPS_DIR = Path(r"D:\ViSpeR\clips")
METADATA_PATH = CLIPS_DIR / "metadata.json"

# Reproducibilidad
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configuración del procesamiento
IMAGE_MODE = "rgb"

# DataLoader
BATCH_SIZE = 4
NUM_WORKERS = 0  # Configuración inicial compatible con Windows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
print("Metadata:", METADATA_PATH)
print("Metadata existe:", METADATA_PATH.exists())

PyTorch: 2.6.0+cu126
CUDA disponible: True
Dispositivo: cuda
Metadata: D:\ViSpeR\clips\metadata.json
Metadata existe: True


In [10]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

clips_df = pd.DataFrame([
    {
        "path": Path(path),
        "label": info["label"],
        "frames": info["length"],
    }
    for path, info in metadata.items()
])

print("Cantidad de clips:", len(clips_df))
display(clips_df.head())

Cantidad de clips: 19527


,path,label,frames
0,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00000.mp4,La ficción es una herramienta poderosa,108
1,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00001.mp4,Una tecnología blanda,82
2,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00002.mp4,Las tecnologías blandas funcionan de la misma...,79
3,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00003.mp4,Cuando en las historias de la Biblia se nos c...,178
4,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00004.mp4,que las cosas buenas son recompensadas y las ...,110


In [11]:
# Normalización experimental: eliminar mayúsculas
clips_df["label_original"] = clips_df["label"]
clips_df["label"] = clips_df["label"].str.lower()

In [12]:
print("Transcripciones vacías:", clips_df["label"].isna().sum())
print("Rutas vacías:", clips_df["path"].isna().sum())
print("Longitudes inválidas:", (clips_df["frames"] <= 0).sum())
print("Archivos inexistentes:", sum(not path.exists() for path in clips_df["path"]))

#assert len(clips_df) == 1304
assert clips_df["label"].notna().all()
assert (clips_df["frames"] > 0).all()
assert all(path.exists() for path in clips_df["path"])

print("\nValidaciones superadas.")

Transcripciones vacías: 0
Rutas vacías: 0
Longitudes inválidas: 0
Archivos inexistentes: 0

Validaciones superadas.


In [13]:
# Detectar archivos inexistentes
missing_mask = ~clips_df["path"].apply(lambda path: path.exists())

missing_clips = clips_df.loc[missing_mask].copy()
existing_clips = clips_df.loc[~missing_mask].copy()

print("RESUMEN")
print("=" * 50)
print("Registros en metadata:", len(clips_df))
print("Archivos inexistentes:", len(missing_clips))
print("Archivos existentes:", len(existing_clips))

# Identificar el video de origen
missing_clips["video_id"] = missing_clips["path"].apply(
    lambda path: path.parent.name
)

print("\nCLIPS INEXISTENTES POR VIDEO")
display(
    missing_clips.groupby("video_id")
    .size()
    .sort_values(ascending=False)
    .to_frame("missing_clips")
)

print("\nARCHIVOS INEXISTENTES")
display(
    missing_clips[["video_id", "path", "label", "frames"]]
    .sort_values(["video_id", "path"])
)

RESUMEN
Registros en metadata: 19527
Archivos inexistentes: 0
Archivos existentes: 19527

CLIPS INEXISTENTES POR VIDEO


,missing_clips
video_id,



ARCHIVOS INEXISTENTES


,video_id,path,label,frames


In [14]:
#cleanear lo que no existe
import json
import shutil
from pathlib import Path

# Backup antes de modificar
backup_path = METADATA_PATH.with_suffix(".json.bak")
shutil.copy2(METADATA_PATH, backup_path)

# Cargar metadata original
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Identificar registros del video problemático
video_id_to_exclude = "5LDd1LodTGM"
KR = 0

keys_to_remove = [
    path
    for path in metadata
    if Path(path).parent.name == video_id_to_exclude
]

print("Registros a eliminar:", len(keys_to_remove))

# Comprobar que sean exactamente los 91 esperados
assert len(keys_to_remove) == KR, (
    f"Se esperaban 91 registros, encontrados: {len(keys_to_remove)}"
)

# Limpiar metadata
for path in keys_to_remove:
    del metadata[path]

# Guardar metadata actualizada
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Metadata actualizada.")
print("Backup:", backup_path)
print("Registros restantes:", len(metadata))

Registros a eliminar: 0
Metadata actualizada.
Backup: D:\ViSpeR\clips\metadata.json.bak
Registros restantes: 19527


In [15]:
all_text = "".join(clips_df["label"].astype(str))

char_counts = Counter(all_text)

vocab_df = pd.DataFrame(
    sorted(char_counts.items()),
    columns=["character", "frequency"]
)

print("Cantidad de caracteres distintos:", len(char_counts))
print("Total de caracteres:", len(all_text))

display(vocab_df)

Cantidad de caracteres distintos: 46
Total de caracteres: 1399427


,character,frequency
0,,249774
1,',5
2,0,807
3,1,538
4,2,413
5,3,206
6,4,155
7,5,264
8,6,118
9,7,108


In [16]:
special_chars = {
    char: count
    for char, count in char_counts.items()
    if not char.isalnum() and not char.isspace()
}

print("Caracteres especiales:", special_chars)
print("Espacios presentes:", " " in char_counts)
print("Mayúsculas presentes:", any(c.isupper() for c in char_counts))
print("Caracteres acentuados:", {
    c: n for c, n in char_counts.items()
    if c.isalpha() and not c.isascii()
})

Caracteres especiales: {"'": 5}
Espacios presentes: True
Mayúsculas presentes: False
Caracteres acentuados: {'ó': 6067, 'í': 8311, 'á': 5617, 'ú': 1268, 'é': 4729, 'ñ': 2121, 'ü': 31, 'à': 1}


In [17]:
# Índice reservado para CTC
BLANK_IDX = 0

# Vocabulario determinista
characters = sorted(char_counts.keys())

char_to_idx = {
    char: idx + 1
    for idx, char in enumerate(characters)
}

idx_to_char = {
    idx: char
    for char, idx in char_to_idx.items()
}

VOCAB_SIZE = len(char_to_idx) + 1  # Incluye blank

print("Caracteres:", len(characters))
print("Tamaño del vocabulario:", VOCAB_SIZE)
print("Índice blank:", BLANK_IDX)

print("\nPrimeros índices:")
for char, idx in list(char_to_idx.items())[:15]:
    print(repr(char), "->", idx)

Caracteres: 46
Tamaño del vocabulario: 47
Índice blank: 0

Primeros índices:
' ' -> 1
"'" -> 2
'0' -> 3
'1' -> 4
'2' -> 5
'3' -> 6
'4' -> 7
'5' -> 8
'6' -> 9
'7' -> 10
'8' -> 11
'9' -> 12
'a' -> 13
'b' -> 14
'c' -> 15


In [18]:
assert BLANK_IDX not in idx_to_char
assert len(char_to_idx) == len(set(char_to_idx))
assert len(idx_to_char) == len(char_to_idx)
assert set(char_to_idx.keys()) == set(char_counts.keys())

print("Vocabulario validado.")

Vocabulario validado.


In [19]:
def encode_text(text):
    unknown_chars = set(text) - set(char_to_idx)

    if unknown_chars:
        raise ValueError(f"Caracteres fuera del vocabulario: {unknown_chars}")

    return [char_to_idx[char] for char in text]

70 caracteres + blank

In [20]:
#prueba

sample_text = clips_df.iloc[0]["label"]

encoded = encode_text(sample_text)

print("Original:", repr(sample_text))
print("Codificada:", encoded)
print("Longitud original:", len(sample_text))
print("Longitud codificada:", len(encoded))

assert len(encoded) == len(sample_text)
assert all(idx != BLANK_IDX for idx in encoded)

print("\nCodificación validada.")

Original: ' la ficción es una herramienta poderosa'
Codificada: [1, 24, 13, 1, 18, 21, 15, 15, 21, 44, 26, 1, 17, 31, 1, 33, 26, 13, 1, 20, 17, 30, 30, 13, 25, 21, 17, 26, 32, 13, 1, 28, 27, 16, 17, 30, 27, 31, 13]
Longitud original: 39
Longitud codificada: 39

Codificación validada.


In [21]:
def decode_tokens(tokens):
    return "".join(
        idx_to_char[idx]
        for idx in tokens
        if idx != BLANK_IDX
    )

In [22]:
decoded = decode_tokens(encoded)

print("Original:   ", repr(sample_text))
print("Decodificada:", repr(decoded))

assert decoded == sample_text

print("\nRound-trip validado.")

Original:    ' la ficción es una herramienta poderosa'
Decodificada: ' la ficción es una herramienta poderosa'

Round-trip validado.


finally 

In [23]:
class ViSpeRDataset(Dataset):
    def __init__(self, dataframe, image_mode):
        if image_mode not in ("grayscale", "rgb"):
            raise ValueError("image_mode debe ser 'grayscale' o 'rgb'")

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_mode = image_mode

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        cap = cv2.VideoCapture(str(row["path"]))

        if not cap.isOpened():
            raise RuntimeError(f"No se pudo abrir: {row['path']}")

        frames = []

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            if self.image_mode == "grayscale":
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                frame = frame[:, :, np.newaxis]
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            raise RuntimeError(f"El clip no contiene frames: {row['path']}")

        video = np.stack(frames).astype(np.float32) / 255.0

        # (T, H, W, C) -> (T, C, H, W)
        video = torch.from_numpy(video).permute(0, 3, 1, 2)

        text = str(row["label"])
        target = torch.tensor(encode_text(text), dtype=torch.long)

        return {
            "video": video,
            "target": target,
            "input_length": video.shape[0],
            "target_length": target.numel(),
            "text": text,
            "path": str(row["path"]),
        }
        

In [24]:
test_dataset = ViSpeRDataset(
    dataframe=clips_df,
    image_mode="rgb"
)

sample = test_dataset[0]

print("Video shape:", sample["video"].shape)
print("Video dtype:", sample["video"].dtype)
print("Video min:", sample["video"].min().item())
print("Video max:", sample["video"].max().item())

print("\nTarget shape:", sample["target"].shape)
print("Input length:", sample["input_length"])
print("Target length:", sample["target_length"])
print("Texto:", repr(sample["text"]))
print("Ruta:", sample["path"])

Video shape: torch.Size([108, 3, 96, 96])
Video dtype: torch.float32
Video min: 0.0
Video max: 1.0

Target shape: torch.Size([39])
Input length: 108
Target length: 39
Texto: ' la ficción es una herramienta poderosa'
Ruta: D:\ViSpeR\clips\videos\0YEeb-WLF2w\00000.mp4


$$
(T,H,W,C)\rightarrow(T,C,H,W)
$$

In [25]:
assert sample["video"].ndim == 4
assert sample["video"].shape[1:] == (3, 96, 96)
assert sample["video"].dtype == torch.float32

assert sample["video"].min() >= 0.0
assert sample["video"].max() <= 1.0

assert sample["input_length"] == sample["video"].shape[0]
assert sample["target_length"] == len(sample["text"])
assert decode_tokens(sample["target"].tolist()) == sample["text"]

print("Todas las validaciones de la muestra pasaron ok")

Todas las validaciones de la muestra pasaron ok


padding temporal por batch

In [26]:
def collate_fn(batch):
    videos = [item["video"] for item in batch]
    targets = [item["target"] for item in batch]

    input_lengths = torch.tensor(
        [video.shape[0] for video in videos],
        dtype=torch.long
    )

    target_lengths = torch.tensor(
        [target.numel() for target in targets],
        dtype=torch.long
    )

    # Longitud temporal máxima del batch
    max_frames = input_lengths.max().item()

    # (B, T_max, C, H, W)
    padded_videos = torch.zeros(
        len(batch),
        max_frames,
        *videos[0].shape[1:],
        dtype=torch.float32
    )

    for i, video in enumerate(videos):
        padded_videos[i, :video.shape[0]] = video

    # CTC espera los targets concatenados
    concatenated_targets = torch.cat(targets)

    return {
        "videos": padded_videos,
        "targets": concatenated_targets,
        "input_lengths": input_lengths,
        "target_lengths": target_lengths,
        "texts": [item["text"] for item in batch],
        "paths": [item["path"] for item in batch],
    }

In [27]:
print(collate_fn)

<function collate_fn at 0x00000234D91ADBD0>


test con 4 clips

In [28]:
test_samples = [
    test_dataset[i]
    for i in range(4)
]

batch = collate_fn(test_samples)

print("Videos:", batch["videos"].shape)
print("Targets:", batch["targets"].shape)
print("Input lengths:", batch["input_lengths"])
print("Target lengths:", batch["target_lengths"])
print("Textos:", batch["texts"])

Videos: torch.Size([4, 178, 3, 96, 96])
Targets: torch.Size([229])
Input lengths: tensor([108,  82,  79, 178])
Target lengths: tensor([ 39,  22,  53, 115])
Textos: [' la ficción es una herramienta poderosa', ' una tecnología blanda', ' las tecnologías blandas funcionan de la misma manera', ' cuando en las historias de la biblia se nos cuenta que hay cosas que son correctas y hay cosas que son incorrectas']


los batch tienen 178 frames porque es la secuencia mas larga ( de estos 4 )

In [29]:
print("Max global:", clips_df["frames"].max())
print("Max del batch:", batch["input_lengths"].max().item())

Max global: 773
Max del batch: 178


In [30]:
# Verificar que el padding sea cero
for i, length in enumerate(batch["input_lengths"]):
    length = length.item()

    assert torch.all(batch["videos"][i, length:] == 0)

# Verificar que los frames originales se conservaron
for i, sample_item in enumerate(test_samples):
    length = sample_item["input_length"]

    assert torch.equal(
        batch["videos"][i, :length],
        sample_item["video"]
    )

# Verificar reconstrucción de cada target
offset = 0

for text, length in zip(batch["texts"], batch["target_lengths"]):
    length = length.item()
    target = batch["targets"][offset:offset + length]

    assert decode_tokens(target.tolist()) == text

    offset += length

assert offset == len(batch["targets"])

print("Padding validado.")
print("Frames originales conservados.")
print("Targets reconstruidos correctamente.")

Padding validado.
Frames originales conservados.
Targets reconstruidos correctamente.


# Crear el dataset 


In [31]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS
)

print("Cantidad de batches:", len(test_loader))
batch = next(iter(test_loader))

print("Videos:", batch["videos"].shape)
print("Targets:", batch["targets"].shape)
print("Input lengths:", batch["input_lengths"])
print("Target lengths:", batch["target_lengths"])

print("\nPrimeras transcripciones:")
for text in batch["texts"]:
    print(repr(text))


Cantidad de batches: 4882
Videos: torch.Size([4, 178, 3, 96, 96])
Targets: torch.Size([229])
Input lengths: tensor([108,  82,  79, 178])
Target lengths: tensor([ 39,  22,  53, 115])

Primeras transcripciones:
' la ficción es una herramienta poderosa'
' una tecnología blanda'
' las tecnologías blandas funcionan de la misma manera'
' cuando en las historias de la biblia se nos cuenta que hay cosas que son correctas y hay cosas que son incorrectas'


# restriccion temporal de CTC 

$$ T_{min} = L + \text{repeticiones consecutivas} $$

In [32]:
def ctc_minimum_length(text):
    repeated_chars = sum(
        text[i] == text[i - 1]
        for i in range(1, len(text))
    )

    return len(text) + repeated_chars

In [33]:
clips_df["ctc_min_frames"] = clips_df["label"].apply(
    ctc_minimum_length
)

clips_df["repeated_pairs"] = (
    clips_df["ctc_min_frames"] - clips_df["label"].str.len()
)

print("Mínimo CTC:")
display(clips_df["ctc_min_frames"].describe())

print("\nPares repetidos consecutivos:")
display(clips_df["repeated_pairs"].describe())

print(
    "\nTranscripción con mayor requisito CTC:",
    repr(clips_df.loc[clips_df["ctc_min_frames"].idxmax(), "label"])
)

print(
    "Máximo requisito temporal:",
    clips_df["ctc_min_frames"].max()
)

Mínimo CTC:


count    19527.000000
mean        72.044144
std         32.226922
min          6.000000
25%         48.000000
50%         68.000000
75%         90.000000
max        350.000000
Name: ctc_min_frames, dtype: float64


Pares repetidos consecutivos:


count    19527.000000
mean         0.377887
std          0.649166
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          5.000000
Name: repeated_pairs, dtype: float64


Transcripción con mayor requisito CTC: ' muchas veces me han preguntado que por qué me dedico a impulsar esta educación hoy estoy clara que impulsar la educación stem es una causa urgente y relevante para toda américa latina pues las competencias stem son el camino para empoderar a las y los jóvenes principalmente a las mujeres si queremos enfrentar como región exitosamente el siglo xxi'
Máximo requisito temporal: 350


La mitad necesita al menos 67 pasos temporales para poder alinear su transcripción.

In [34]:
clips_df["temporal_margin"] = (
    clips_df["frames"] - clips_df["ctc_min_frames"]
)

clips_df["min_temporal_ratio"] = (
    clips_df["ctc_min_frames"] / clips_df["frames"]
)

print("Clips que no cumplen el mínimo ni siquiera antes del encoder:",
      (clips_df["temporal_margin"] < 0).sum())

print("\nMargen temporal:")
display(clips_df["temporal_margin"].describe())

print("\nProporción mínima de pasos que deben conservarse:")
display(clips_df["min_temporal_ratio"].describe())

print("\nClip con menor margen:")
display(
    clips_df.loc[
        clips_df["temporal_margin"].idxmin(),
        ["label", "frames", "ctc_min_frames", "temporal_margin"]
    ]
)

Clips que no cumplen el mínimo ni siquiera antes del encoder: 204

Margen temporal:


count    19527.000000
mean        51.748707
std         36.042447
min        -38.000000
25%         26.000000
50%         44.000000
75%         69.000000
max        486.000000
Name: temporal_margin, dtype: float64


Proporción mínima de pasos que deben conservarse:


count    19527.000000
mean         0.602682
std          0.159167
min          0.030151
25%          0.496855
50%          0.596567
75%          0.701809
max          1.545455
Name: min_temporal_ratio, dtype: float64


Clip con menor margen:


label               digo me puedes poner una campanilla te puedo ...
frames                                                            78
ctc_min_frames                                                   116
temporal_margin                                                  -38
Name: 12632, dtype: object

Ohsh : 17 clips no pueden pasar por la STNN 

In [35]:
problematic_clips = clips_df[
    clips_df["temporal_margin"] < 0
].copy()

print("Cantidad de clips problemáticos:", len(problematic_clips))

display(
    problematic_clips[
        ["path", "label", "frames", "ctc_min_frames", "temporal_margin"]
    ]
)

Cantidad de clips problemáticos: 204


,path,label,frames,ctc_min_frames,temporal_margin
24,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00024.mp4,esto es lo que hemos visto en las películas y...,50,60,-10
42,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00042.mp4,y en esa época se estaba empezando a hablar d...,61,70,-9
71,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00071.mp4,pueden servir para entender lo que no somos c...,63,64,-1
300,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00013.mp4,en un momento dado y esto lo podéis ver en in...,70,78,-8
303,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00016.mp4,ayer por la noche vi que la capitalización bu...,54,63,-9
...,...,...,...,...,...
19228,D:\ViSpeR\clips\videos\kreVAvbU0JI\00035.mp4,y aquí vino el inverso de lo que les he dicho,43,46,-3
19321,D:\ViSpeR\clips\videos\KofVoozWwD0\00007.mp4,habéis tenido una falta de motivación inmensa...,101,117,-16
19354,D:\ViSpeR\clips\videos\icS-lXrRSeY\00007.mp4,el tema es que si esta adulto adulto adulto a...,79,86,-7
19357,D:\ViSpeR\clips\videos\icS-lXrRSeY\00010.mp4,sí los voy a invitar a que nos pongamos un po...,64,65,-1


In [36]:
print("Proporción mínima de pasos que deben conservarse:")
display(clips_df["min_temporal_ratio"].describe())

Proporción mínima de pasos que deben conservarse:


count    19527.000000
mean         0.602682
std          0.159167
min          0.030151
25%          0.496855
50%          0.596567
75%          0.701809
max          1.545455
Name: min_temporal_ratio, dtype: float64

In [37]:
problematic_clips = problematic_clips.copy()

problematic_clips["duration_seconds"] = (
    problematic_clips["frames"] / 25
)

problematic_clips["chars_per_second"] = (
    problematic_clips["label"].str.len()
    / problematic_clips["duration_seconds"]
)

problematic_clips["video_id"] = (
    problematic_clips["path"].apply(lambda p: p.parent.name)
)

display(
    problematic_clips[
        [
            "video_id",
            "path",
            "frames",
            "duration_seconds",
            "label",
            "ctc_min_frames",
            "temporal_margin",
            "chars_per_second"
        ]
    ]
)

,video_id,path,frames,duration_seconds,label,ctc_min_frames,temporal_margin,chars_per_second
24,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00024.mp4,50,2.00,esto es lo que hemos visto en las películas y...,60,-10,30.000000
42,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00042.mp4,61,2.44,y en esa época se estaba empezando a hablar d...,70,-9,28.688525
71,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00071.mp4,63,2.52,pueden servir para entender lo que no somos c...,64,-1,25.396825
300,kOq_GFTRWxg,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00013.mp4,70,2.80,en un momento dado y esto lo podéis ver en in...,78,-8,27.857143
303,kOq_GFTRWxg,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00016.mp4,54,2.16,ayer por la noche vi que la capitalización bu...,63,-9,29.166667
...,...,...,...,...,...,...,...,...
19228,kreVAvbU0JI,D:\ViSpeR\clips\videos\kreVAvbU0JI\00035.mp4,43,1.72,y aquí vino el inverso de lo que les he dicho,46,-3,26.744186
19321,KofVoozWwD0,D:\ViSpeR\clips\videos\KofVoozWwD0\00007.mp4,101,4.04,habéis tenido una falta de motivación inmensa...,117,-16,28.960396
19354,icS-lXrRSeY,D:\ViSpeR\clips\videos\icS-lXrRSeY\00007.mp4,79,3.16,el tema es que si esta adulto adulto adulto a...,86,-7,27.215190
19357,icS-lXrRSeY,D:\ViSpeR\clips\videos\icS-lXrRSeY\00010.mp4,64,2.56,sí los voy a invitar a que nos pongamos un po...,65,-1,25.390625


In [38]:
for idx, row in problematic_clips.iterrows():
    print(f"\nClip: {row['path'].name}")
    print(f"Video: {row['video_id']}")
    print(f"Duración: {row['duration_seconds']:.2f} s")
    print(f"Frames: {row['frames']}")
    print(f"Caracteres: {len(row['label'])}")
    print(f"Palabras: {len(row['label'].split())}")
    print(f"Requisito CTC: {row['ctc_min_frames']}")
    print(f"Transcripción: {repr(row['label'])}")
    print("-" * 80)


Clip: 00024.mp4
Video: 0YEeb-WLF2w
Duración: 2.00 s
Frames: 50
Caracteres: 60
Palabras: 13
Requisito CTC: 60
Transcripción: ' esto es lo que hemos visto en las películas y en los libros'
--------------------------------------------------------------------------------

Clip: 00042.mp4
Video: 0YEeb-WLF2w
Duración: 2.44 s
Frames: 61
Caracteres: 70
Palabras: 14
Requisito CTC: 70
Transcripción: ' y en esa época se estaba empezando a hablar del internet de las cosas'
--------------------------------------------------------------------------------

Clip: 00071.mp4
Video: 0YEeb-WLF2w
Duración: 2.52 s
Frames: 63
Caracteres: 64
Palabras: 11
Requisito CTC: 64
Transcripción: ' pueden servir para entender lo que no somos capaces de entender'
--------------------------------------------------------------------------------

Clip: 00013.mp4
Video: kOq_GFTRWxg
Duración: 2.80 s
Frames: 70
Caracteres: 78
Palabras: 15
Requisito CTC: 78
Transcripción: ' en un momento dado y esto lo podéis ver en internet 

In [39]:
clips_df["duration_seconds"] = clips_df["frames"] / 25

clips_df["words_per_second"] = (
    clips_df["label"].str.split().str.len()
    / clips_df["duration_seconds"]
)

print("Velocidad de habla estimada (palabras/s):")
display(clips_df["words_per_second"].describe())

print("\nPercentiles:")
display(
    clips_df["words_per_second"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
)

print("\nRelación entre requisito CTC y frames:")
display(clips_df["min_temporal_ratio"].describe())

Velocidad de habla estimada (palabras/s):


count    19527.000000
mean         2.694753
std          0.810833
min          0.251256
25%          2.147239
50%          2.631579
75%          3.174603
max          8.522727
Name: words_per_second, dtype: float64


Percentiles:


0.50    2.631579
0.75    3.174603
0.90    3.719008
0.95    4.104478
0.99    4.963411
1.00    8.522727
Name: words_per_second, dtype: float64


Relación entre requisito CTC y frames:


count    19527.000000
mean         0.602682
std          0.159167
min          0.030151
25%          0.496855
50%          0.596567
75%          0.701809
max          1.545455
Name: min_temporal_ratio, dtype: float64

In [40]:
clips_df["video_id"] = clips_df["path"].apply(lambda p: p.parent.name)

clips_df["is_ctc_problematic"] = clips_df["temporal_margin"] < 0

video_stats = (
    clips_df.groupby("video_id")
    .agg(
        clips=("path", "count"),
        mean_words_per_second=("words_per_second", "mean"),
        max_words_per_second=("words_per_second", "max"),
        problematic_clips=("is_ctc_problematic", "sum"),
    )
    .sort_values("problematic_clips", ascending=False)
)

display(video_stats)

,clips,mean_words_per_second,max_words_per_second,problematic_clips
video_id,,,,
dvmJ_HyfBto,101,3.466275,8.522727,10
mukpAM7t85A,42,3.958522,6.666667,10
L29ItpXnzW4,41,3.612905,6.097561,8
NNPH7KWjrrs,139,2.964839,5.312500,6
LoJMONJTAHs,29,3.671214,5.882353,6
...,...,...,...,...
MeJxFi9dg-M,20,3.351345,5.084746,0
MMPZtbJk9AE,31,2.998862,4.562044,0
MEwoB-K8DE0,80,3.310679,5.182927,0


In [41]:
print("Cantidad de clips según margen temporal:")

for threshold in [0, 5, 10, 15, 20, 30]:
    count = (clips_df["temporal_margin"] <= threshold).sum()
    print(f"Margen <= {threshold:2d}: {count:4d} clips")

print("\nClips con menor margen:")
display(
    clips_df.sort_values("temporal_margin")[
        [
            "video_id",
            "path",
            "frames",
            "ctc_min_frames",
            "temporal_margin",
            "words_per_second",
            "label"
        ]
    ].head(30)
)

Cantidad de clips según margen temporal:
Margen <=  0:  236 clips
Margen <=  5:  538 clips
Margen <= 10: 1104 clips
Margen <= 15: 1984 clips
Margen <= 20: 3221 clips
Margen <= 30: 6093 clips

Clips con menor margen:


,video_id,path,frames,ctc_min_frames,temporal_margin,words_per_second,label
12632,xkJKAdWdF4c,D:\ViSpeR\clips\videos\xkJKAdWdF4c\00025.mp4,78,116,-38,5.769231,digo me puedes poner una campanilla te puedo ...
3918,dvmJ_HyfBto,D:\ViSpeR\clips\videos\dvmJ_HyfBto\00070.mp4,66,98,-32,7.575758,hace que en el momento en el que tienes un pr...
2660,mukpAM7t85A,D:\ViSpeR\clips\videos\mukpAM7t85A\00035.mp4,99,123,-24,6.313131,y si hablamos de que va a haber problemas en ...
3905,dvmJ_HyfBto,D:\ViSpeR\clips\videos\dvmJ_HyfBto\00057.mp4,44,68,-24,8.522727,yo voy a hacer esto y voy a hacer esto y organ...
13012,AG40mZmYj40,D:\ViSpeR\clips\videos\AG40mZmYj40\00071.mp4,79,102,-23,5.696203,ellos no quieren ser el primero de la fila t...
2656,mukpAM7t85A,D:\ViSpeR\clips\videos\mukpAM7t85A\00031.mp4,68,88,-20,5.882353,bueno pues el problema que tenemos es que esa...
2752,LQCAV2Q5j5c,D:\ViSpeR\clips\videos\LQCAV2Q5j5c\00038.mp4,56,74,-18,6.250000,se entienda lo que se entienda por fracaso es...
12386,QuVbCnov33I,D:\ViSpeR\clips\videos\QuVbCnov33I\00026.mp4,41,59,-18,7.926829,un saludo y un beso se lo decimos a los chico...
9870,ig89u6RgppU,D:\ViSpeR\clips\videos\ig89u6RgppU\00039.mp4,45,62,-17,7.222222,soy autónomo o sea que tengo que pensar el pl...
12625,xkJKAdWdF4c,D:\ViSpeR\clips\videos\xkJKAdWdF4c\00018.mp4,58,75,-17,6.465517,con lo cual yo no podría estar delante de la ...


In [42]:
for threshold in [0, 5, 10, 15, 20, 30]:
    count = (clips_df["temporal_margin"] <= threshold).sum()
    percentage = count / len(clips_df) * 100
    print(f"Margen <= {threshold:2d}: {count:4d} clips ({percentage:.2f}%)")

Margen <=  0:  236 clips (1.21%)
Margen <=  5:  538 clips (2.76%)
Margen <= 10: 1104 clips (5.65%)
Margen <= 15: 1984 clips (10.16%)
Margen <= 20: 3221 clips (16.50%)
Margen <= 30: 6093 clips (31.20%)


15 clips de 1300 con margen nulo o negativo y solo 1 igual a 0   
restringe la arqui de la red a 3 cosas:   
1. el tiempo de los clips no se puede reducir mucho 
2. calcular longitudes temporales despeus del encoder, los originales antes de CTCLoss no van a servir 
3. no podemos descartar el 33% del dataset 

### Simulacion 

In [43]:
reduction_factors = [1, 2, 3, 4]

simulation_results = []

for factor in reduction_factors:
    clips_df[f"frames_after_x{factor}"] = np.ceil(
        clips_df["frames"] / factor
    ).astype(int)

    clips_df[f"margin_after_x{factor}"] = (
        clips_df[f"frames_after_x{factor}"]
        - clips_df["ctc_min_frames"]
    )

    invalid = (clips_df[f"margin_after_x{factor}"] < 0).sum()

    simulation_results.append({
        "reduction_factor": factor,
        "remaining_clips": len(clips_df) - invalid,
        "invalid_clips": invalid,
        "invalid_percentage": invalid / len(clips_df) * 100,
        "minimum_output_frames": clips_df[f"frames_after_x{factor}"].min(),
        "median_output_frames": clips_df[f"frames_after_x{factor}"].median(),
    })

reduction_df = pd.DataFrame(simulation_results)

display(reduction_df)

,reduction_factor,remaining_clips,invalid_clips,invalid_percentage,minimum_output_frames,median_output_frames
0,1,19323,204,1.044707,40,113.0
1,2,5326,14201,72.724945,20,57.0
2,3,789,18738,95.959441,14,38.0
3,4,281,19246,98.560967,10,29.0


una reducción temporal de 2x seria re agresiva. incompatible con el 70% de los clips 

In [44]:
import pyarrow

clips_df_to_save = clips_df.copy()
clips_df_to_save["path"] = clips_df_to_save["path"].astype(str)

clips_df_to_save.to_parquet(
    "data/clips_metadata.parquet",
    index=False
)

with open("data/vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(char_to_idx, f, ensure_ascii=False, indent=2)


In [45]:
hours = clips_df["frames"].sum() / 25 / 3600
n_vid = clips_df["video_id"].nunique()
print(f"{n_vid} videos | {hours:.2f} h | {hours / n_vid * 60:.1f} min de clips por video | "
      f"{len(clips_df) / n_vid:.0f} clips por video")

488 videos | 26.86 h | 3.3 min de clips por video | 40 clips por video
